## Importar Librerías

In [1]:
import requests
import openai
from datetime import datetime
import getpass
import os

## Se configurará la memoria del chat, mediante LangChain

In [2]:
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

Enter your OpenAI API key:  ········


In [3]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4o")

In [4]:
from langchain.schema import AIMessage, HumanMessage, SystemMessage

chat_history = []
if not chat_history:
  system_message = SystemMessage(content='Eres un asistente médico, capacitado para responder a las preguntas acerca de los pacientes')
  chat_history.append(system_message)

## Funciones

In [5]:
FHIR_BASE_URL = "http://localhost:8080/fhir"

RESOURCE_CONFIG = {
    "Condition": {
        "label": "Condiciones",
        "extractor": lambda r: r["resource"]["code"].get("text", "Sin descripción")
    },
    "Observation": {
        "label": "Observaciones",
        "extractor": lambda r: f'{r["resource"]["code"].get("text", "Sin nombre")}: ' +
                               f'{r["resource"].get("valueQuantity", {}).get("value", "N/A")} ' +
                               f'{r["resource"].get("valueQuantity", {}).get("unit", "")}'
    },
    "Encounter": {
        "label": "Agendamientos",
        "extractor": lambda r: (
            f"Estado: {r['resource'].get('status', 'Desconocido')} | "
            f"Clase: {r['resource'].get('class', {}).get('code', 'Sin clase')} | "
            f"Motivo: {r['resource'].get('type', [{}])[0].get('text', 'Sin motivo')} | "
            f"Inicio: {r['resource'].get('period', {}).get('start', 'Fecha no disponible')} | "
            f"Ubicación: {r['resource'].get('location', [{}])[0].get('location', {}).get('display', 'No especificada')} | "
            f"Profesional: {r['resource'].get('participant', [{}])[0].get('individual', {}).get('display', 'No asignado')}"
        )
    },
    "MedicationRequest": {
        "label": "Medicamentos recetados",
        "extractor": lambda r: (
            r["resource"].get("medicationCodeableConcept", {}).get("text") or
            r["resource"].get("medicationReference", {}).get("display") or
            "Desconocido")
    }
}

def get_patients():
    res = requests.get(f"{FHIR_BASE_URL}/Patient")
    return res.json()["entry"] if res.ok else []

def get_patient_id(patients):
    return patients[0]["resource"]["id"]

def get_resources(resource_type, patient_id):
    url = f"{FHIR_BASE_URL}/{resource_type}?patient={patient_id}"
    res = requests.get(url)
    return res.json().get("entry", []) if res.ok else []

def create_summary(patient_id):
    summary = f"Resumen clínico del paciente ID: {patient_id}\n\n"
    for resource_type, config in RESOURCE_CONFIG.items():
        entries = get_resources(resource_type, patient_id)
        if entries:
            summary += f"{config['label']}:\n"
            for r in entries[:5]:
                summary += f" - {config['extractor'](r)}\n"
            summary += "\n"
    return summary.strip()

def ask_gpt(summary):
    # prompt = f"Explica de forma simple el siguiente resumen clínico:\n\n{summary}"
    prompt = f"Explica de forma simple el siguiente resumen clínico:\n\n{summary}\n Deja indicaciones médicas al paciente"

    # Usamos el endpoint correcto para chat
    chat_history.append(HumanMessage(content=prompt))
    response = model.invoke(chat_history).content
    chat_history.append(AIMessage(content=response))
       
    # Retornamos el texto generado
    return response

def preguntar(consulta):
    prompt = f"De acuerdo a lo que sabes del paciente, contesta la pregunta:\n\n{consulta}"
    
    # Usamos langChain
    chat_history.append(HumanMessage(content=prompt))
    response = model.invoke(chat_history).content
    chat_history.append(AIMessage(content=response))
    
    # Retornamos el texto generado
    return response

## Comenzar a trabajar con los datos rescatados desde FHIR

In [6]:
patients = get_patients()
sexo = patients[0]["resource"]["gender"]
nacimiento = datetime.strptime(patients[1]["resource"]["birthDate"],'%Y-%m-%d')
hoy = datetime.now()
edad = hoy.year - nacimiento.year - ((hoy.month, hoy.day) < (nacimiento.month, nacimiento.day))

In [7]:
patients = get_patients()
patient_id = get_patient_id(patients)
summary = create_summary(patient_id)
print(summary)

Resumen clínico del paciente ID: 496

Condiciones:
 - Received higher education (finding)
 - Transport problem (finding)
 - Lack of access to transportation (finding)
 - Body mass index 30+ - obesity (finding)
 - Misuses drugs (finding)

Observaciones:
 - Body Height: 174.5 cm
 - Pain severity - 0-10 verbal numeric rating [Score] - Reported: 2 {score}
 - Body Weight: 91.5 kg
 - Body mass index (BMI) [Ratio]: 30.04 kg/m2
 - Blood pressure panel with all children optional: N/A 

Agendamientos:
 - Estado: finished | Clase: AMB | Motivo: General examination of patient (procedure) | Inicio: 1995-09-10T15:11:02-04:00 | Ubicación: DAVIS SQUARE FAMILY PRACTICE | Profesional: Dr. Vito638 Barton704
 - Estado: finished | Clase: AMB | Motivo: General examination of patient (procedure) | Inicio: 1999-09-19T15:11:02-04:00 | Ubicación: DAVIS SQUARE FAMILY PRACTICE | Profesional: Dr. Vito638 Barton704
 - Estado: finished | Clase: AMB | Motivo: General examination of patient (procedure) | Inicio: 2002-

## Usar la API de OpenAI para poder contextualizar el caso

In [8]:
explicacion = ask_gpt(summary)
print(explicacion)

Este resumen clínico corresponde al paciente ID: 496 y contiene la siguiente información:

**Condiciones Médicas:**
1. El paciente ha recibido educación superior.
2. Tiene problemas de transporte y falta de acceso al mismo.
3. Presenta obesidad con un índice de masa corporal (IMC) de más de 30.
4. Hace uso indebido de drogas.

**Observaciones Físicas:**
- Altura: 174.5 cm
- Peso: 91.5 kg
- IMC calculado: 30.04 kg/m², lo que clasifica al paciente como obeso.
- Reportó un nivel de dolor leve (2 sobre 10 en una escala de dolor).
- No hay datos disponibles sobre la presión arterial.

**Historial de Citas:**
El paciente ha tenido varias consultas médicas para chequeos generales en diferentes fechas desde 1995 hasta 2015, principalmente en la práctica familiar de Davis Square y en el Spaulding Hospital for Continuing Care.

**Medicamentos:**
El paciente está prescrito con Simvastatin, que se utiliza generalmente para controlar el colesterol y prevenir enfermedades cardiovasculares. Hubo una 

## Ahora la idea es poder hacer una pregunta libre

In [9]:
respuesta = preguntar("Cual debería ser la siguiente interconsulta para este paciente?")
print (respuesta)

Teniendo en cuenta la información del resumen clínico, la siguiente interconsulta recomendada para el paciente podría ser con un especialista en control de peso o un endocrinólogo. Esto se debe al IMC elevado y la condición de obesidad que presenta el paciente. Un endocrinólogo podría ayudar a abordar otros posibles problemas metabólicos subyacentes y proporcionar un plan de tratamiento más ajustado para manejar el peso y mejorar la salud general.

Además, dado el uso indebido de drogas, sería recomendable también una interconsulta con un especialista en salud mental o un especialista en adicciones para evaluar y tratar adecuadamente este comportamiento, ya que podría tener implicaciones significativas en la salud global del paciente.

Por último, abordar el problema de acceso al transporte podría requerir la ayuda de un trabajador social o un coordinador de atención que pueda conectar al paciente con los recursos comunitarios necesarios para mejorar su acceso a los servicios de salud.

In [10]:
respuesta = preguntar("Que exámenes debe de hacerse el paciente?")
print (respuesta)

Dado el perfil clínico del paciente, sería recomendable llevar a cabo los siguientes exámenes:

1. **Perfil Lipídico Completo:** Para evaluar los niveles de colesterol total, LDL, HDL y triglicéridos, especialmente porque está tomando Simvastatin, un medicamento que se utiliza para controlar los niveles de colesterol.

2. **Evaluación de la Función Hepática:** Puesto que las estatinas pueden afectar el hígado, es prudente monitorear la función hepática periódicamente.

3. **Nivel de Glucosa en Ayunas y Hemoglobina A1c:** Para descartar o detectar precozmente diabetes tipo 2, dado el IMC elevado y el riesgo aumentado asociado con la obesidad.

4. **Exámenes de Función Renal:** Para asegurar que los riñones están funcionando adecuadamente, ya que tanto la obesidad como el uso de ciertas medicaciones pueden afectar la función renal.

5. **Pruebas de Detección de Uso de Drogas:** Si el paciente está de acuerdo, esto puede ayudar a documentar y guiar el tratamiento para el uso indebido de d

In [11]:
respuesta = preguntar("El paciente requiere un equipo multi disciplinario? Qué especialistas deben estar en este equipo?")
print (respuesta)

Sí, considerando las múltiples condiciones presentes en el paciente, sería beneficioso contar con un enfoque de equipo multidisciplinario para proporcionar un cuidado integral. Los especialistas que podrían formar parte de este equipo incluyen:

1. **Médico de Atención Primaria:** Para coordinar el cuidado general del paciente y supervisar el manejo de sus condiciones crónicas.

2. **Endocrinólogo:** Para tratar y manejar la obesidad y cualquier problema metabólico o endocrino relacionado, como el riesgo de diabetes.

3. **Nutricionista o Dietista:** Para desarrollar y supervisar un plan de alimentación saludable que aborde la pérdida de peso y mejore la salud general del paciente.

4. **Especialista en Salud Mental o Psiquiatra:** Para abordar el uso indebido de drogas y cualquier posible trastorno psicológico o emocional.

5. **Consejero o Terapeuta de Adicciones:** Para proporcionar apoyo y tratamiento específico para el uso indebido de drogas.

6. **Cardiólogo:** Podría ser necesar